# Recommender: inspect each stage

Run cells from top to bottom. HF lookup, Tavily search, and Nemotron validation are separate calls that share saved evidence. The notebook uses the existing Linux `.venv` through WSL when the Jupyter kernel is on Windows. It does not install packages or launch a benchmark. Keep API keys in the repository `.env`, not in this notebook.

In [1]:
from pathlib import Path
from datetime import datetime, timezone
import json, os, shutil, sys

repo_root = next((p for p in (Path.cwd(), Path.cwd().parent) if (p / 'config/recommender.yaml').exists()), None)
assert repo_root is not None, 'Open the notebook from the repository or notebooks/ folder'
linux_python = repo_root / '.venv/bin/python'
windows_python = repo_root / '.venv/Scripts/python.exe'
if os.name == 'nt' and linux_python.exists():
    assert shutil.which('wsl.exe'), 'WSL is needed to run the Linux .venv'
    assert repo_root.drive, 'Use a WSL kernel for a repository on a UNC path'
    wsl_root = f'/mnt/{repo_root.drive[0].lower()}/{repo_root.relative_to(repo_root.anchor).as_posix()}'
    python_runner = f'wsl.exe --cd "{wsl_root}" -- ./.venv/bin/python'
elif os.name == 'nt' and windows_python.exists():
    python_runner = r'.\.venv\Scripts\python.exe'
else:
    # Use this notebook's selected WSL kernel, not the project's GPU .venv.
    python_runner = f'env PYTHONPATH=src {sys.executable}'
model_id = 'Qwen/Qwen3.5-4B'
task_request = 'Find a good model with a speculative decoding head for one 80GB A100 GPU, a budget of $5/hour, and a maximum of 2000 generated tokens; any license, any framework, sweep mode.'
demo_dir = Path('artifacts/recommendations') / f"notebook-{datetime.now(timezone.utc):%Y%m%d-%H%M%S}"
demo_dir_cli = demo_dir.as_posix()
print('Model:', model_id, '\nOutput:', repo_root / demo_dir)

Model: Qwen/Qwen3.5-4B 
Output: /mnt/c/Users/maorb/work/hf_validator/artifacts/recommendations/notebook-20260918-115434


## Run 2

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json, os, shutil, sys

repo_root = next((p for p in (Path.cwd(), Path.cwd().parent) if (p / 'config/recommender.yaml').exists()), None)
assert repo_root is not None, 'Open the notebook from the repository or notebooks/ folder'
linux_python = repo_root / '.venv/bin/python'
windows_python = repo_root / '.venv/Scripts/python.exe'
if os.name == 'nt' and linux_python.exists():
    assert shutil.which('wsl.exe'), 'WSL is needed to run the Linux .venv'
    assert repo_root.drive, 'Use a WSL kernel for a repository on a UNC path'
    wsl_root = f'/mnt/{repo_root.drive[0].lower()}/{repo_root.relative_to(repo_root.anchor).as_posix()}'
    python_runner = f'wsl.exe --cd "{wsl_root}" -- ./.venv/bin/python'
elif os.name == 'nt' and windows_python.exists():
    python_runner = r'.\.venv\Scripts\python.exe'
else:
    # Use this notebook's selected WSL kernel, not the project's GPU .venv.
    python_runner = f'env PYTHONPATH=src {sys.executable}'
model_id = 'Qwen/Qwen3.5-4B'
task_request = 'Find a good model for social network analysis with a speculative decoding head for one 80GB A100 GPU, a budget of $5/hour, and a maximum of 2000 generated tokens; any license, any framework, sweep mode.'
demo_dir = Path('artifacts/recommendations') / f"notebook-{datetime.now(timezone.utc):%Y%m%d-%H%M%S}"
demo_dir_cli = demo_dir.as_posix()
print('Model:', model_id, '\nOutput:', repo_root / demo_dir)

Model: Qwen/Qwen3.5-4B 
Output: /mnt/c/Users/maorb/work/hf_validator/artifacts/recommendations/notebook-20260918-084329


In [3]:
%cd {repo_root}

/mnt/c/Users/maorb/work/hf_validator


## 1. Check keys

This checks the **WSL process** and repository `.env` without printing secrets. Nemotron needs `NEBIUS_API_KEY`; Tavily needs `TAVILY_API_KEY`. If Tavily says `false`, add the key to `.env` and rerun this cell.

In [4]:
!{python_runner} -m hf_model_watch.recommender.probe status --output-dir {demo_dir_cli}

{"nebius_key_present": true, "tavily_key_present": true, "output_dir": "artifacts/recommendations/notebook-20260918-084329"}


## 2. Hugging Face scan

Look up one exact model. The cell shows the revision, license, parameters, context, architecture, chat-template presence, size, missing fields, and card path. The complete scan record is saved as `hf_record.json`. If this cell fails, stop here: later cells need its artifact.

In [ ]:
!{python_runner} -m hf_model_watch.recommender.probe hf --model-id {model_id} --output-dir Path('artifacts/recommendations') / "social_network"

{"model_id": "Qwen/Qwen3.5-4B", "revision": "851bf6e806efd8d0a36b00ddf55e13ccb7b8cd0a", "license": "apache-2.0", "parameters_billion": 4.66, "context_length": 2048, "architecture": ["Qwen3_5ForConditionalGeneration"], "chat_template_present": true, "download_size_gb": 8.68, "missing_fields": [], "card_artifact": "artifacts/recommendations/notebook-20260918-084329/Qwen_Qwen3.5-4B-851bf6e806efd8d0a36b00ddf55e13ccb7b8cd0a.README.md", "record_artifact": "artifacts/recommendations/notebook-20260918-084329/hf_record.json"}


## 3. Tavily research

Run focused searches for fields HF did not establish, plus speculative decoding and runtime evidence. The following cell displays every saved query and result, including source text and URLs. The data is saved in `tavily_searches.json` and `tavily_evidence.json`. A missing key produces an empty evidence file.

In [6]:
!{python_runner} -m hf_model_watch.recommender.probe tavily --output-dir {demo_dir_cli} --request "{task_request}"

{"status": "complete", "cases": [{"case": "exact", "query": "\"Qwen/Qwen3.5-4B\"", "count": 8}, {"case": "draft_pair", "query": "\"Qwen3.5-4B\" DFlash draft target site:huggingface.co", "count": 8}, {"case": "speculative", "query": "\"Qwen/Qwen3.5-4B\" Medusa EAGLE D-Spark D-Flash draft heads runtime", "count": 8}, {"case": "runtime", "query": "\"Qwen/Qwen3.5-4B\" vLLM SGLang TRT-LLM compatibility", "count": 8}, {"case": "origin", "query": "\"Qwen/Qwen3.5-4B\" base_model parent model", "count": 8}], "source_count": 40, "top_sources": [{"url": "https://recipes.vllm.ai/Qwen/Qwen3.5-4B", "claim": "vLLMvLLM/Recipes\n\nQwen\n\n# Qwen/Qwen3.5-4B\n\nQwen3.5 compact dense multimodal model (4B) \u2014 fits on 16 GB consumer GPUs with full 262K context or one Xeon 6 NUMA node or Intel Arc Pro B60/B70\n\nConsumer-GPU-friendly Qwen3.5 dense with MTP support\n\nView on HuggingFace\n\ndense4B262,144 ctxvLLM 0.17.0+multi"}, {"url": "https://apxml.com/models/qwen35-4b", "claim": "#117\n\nCoding Rank\n

In [7]:
from html import escape
from IPython.display import HTML, display

searches_path = repo_root / demo_dir / 'tavily_searches.json'
evidence_path = repo_root / demo_dir / 'tavily_evidence.json'
if not evidence_path.exists():
    raise FileNotFoundError(f'Run the Tavily cell first: {evidence_path}')
searches = json.loads(searches_path.read_text(encoding='utf-8')) if searches_path.exists() else []
sources = json.loads(evidence_path.read_text(encoding='utf-8'))
if not searches:
    display(HTML('<p>No searches ran. Check the Tavily status output above.</p>'))
for number, search in enumerate(searches, 1):
    matches = [row for row in sources if row.get('research_case') == search['case']]
    title = f'{number}. {search["case"]}: {search["query"]}'
    display(HTML(f'<h3>{escape(title)}</h3><p>{len(matches)} saved results' + (f'; error: {escape(search["error"])}' if search.get('error') else '') + '</p>'))
    for index, row in enumerate(matches, 1):
        url = str(row.get('source_url') or '')
        label = escape(str(row.get('title') or url))
        body = escape(str(row.get('retrieved_text') or 'No source text returned.'))
        display(HTML(f'<details><summary>{index}. <a href="{escape(url, quote=True)}" target="_blank">{label}</a> (score: {escape(str(row.get("score")))})</summary><pre style="white-space:pre-wrap;overflow-wrap:anywhere">{body}</pre></details>'))


## 4. Nemotron validation

This call uses the **saved** HF and Tavily artifacts. The next cell displays the full Nemotron reasoning, final JSON, cited URLs, and the final recommendation. This example uses the stated $5/hour A100 price assumption; replace it with your actual rental price before deployment. A documented drafter pair can pass 80 GB capacity while A100 speedup remains unmeasured.

In [8]:
!{python_runner} -m hf_model_watch.recommender.probe nemotron --output-dir {demo_dir_cli} --request "{task_request}" --gpu-memory-gb 80 --budget-usd-per-hour 5 --gpu-hourly-cost-usd 5 --license any --framework-mode sweep

{"status": "recommended", "model": "Qwen/Qwen3.5-4B", "router_assessment": {"viable_model_ids": ["Qwen/Qwen3.5-4B"], "insufficient_evidence": false, "reasoning": "Model fits 80GB A100, download size 8.68GB, license Apache-2.0, speculative decoding head available via z-lab/Qwen3.5-4B-DFlash draft paired with target, enabling speculative decoding within budget.", "field_assessments": {"Qwen/Qwen3.5-4B": {"base_model": {"value": "Qwen/Qwen3.5-4B-Base", "status": "unknown", "confidence": 0.5, "sources": ["https://huggingface.co/Qwen/Qwen3.5-4B/tree/851bf6e806efd8d0a36b00ddf55e13ccb7b8cd0a"], "action": "keep_blocked"}, "parameter_count": {"value": 4.66, "status": "verified", "confidence": 0.95, "sources": ["https://huggingface.co/Qwen/Qwen3.5-4B/tree/851bf6e806efd8d0a36b00ddf55e13ccb7b8cd0a"], "action": "accept"}, "config": {"value": {"model_type": "qwen3_5", "architectures": ["Qwen3_5ForConditionalGeneration"], "max_position_embeddings": null}, "status": "verified", "confidence": 0.95, "so

In [9]:
from IPython.display import Markdown, display

response_path = repo_root / demo_dir / 'nemotron_validation_response.json'
if not response_path.exists():
    raise FileNotFoundError(f'Run the Nemotron cell first: {response_path}')
response = json.loads(response_path.read_text(encoding='utf-8'))
message = response.get('message', response)  # supports artifacts from earlier notebook versions
reasoning = message.get('reasoning') or message.get('reasoning_content')
final_answer = message.get('content')

display(Markdown('## Nemotron reasoning'))
display(Markdown(str(reasoning) if reasoning else '*No reasoning returned by the API.*'))
try:
    answer = json.loads(final_answer)
except (TypeError, ValueError) as exc:
    answer = {}
    display(Markdown(f'## Invalid Nemotron answer\nThe API returned text that is not one JSON object: `{exc}`. The complete raw answer is saved in `{response_path}`.'))
    display(HTML(f'<details><summary>Show raw answer</summary><pre style="white-space:pre-wrap;overflow-wrap:anywhere">{escape(str(final_answer or ""))}</pre></details>'))
else:
    display(Markdown('## Final answer'))
    display(Markdown(f'```json\n{json.dumps(answer, indent=2, ensure_ascii=False)}\n```'))
saved_sources = json.loads((repo_root / demo_dir / 'tavily_evidence.json').read_text(encoding='utf-8'))
saved_urls = {row.get('source_url') for row in saved_sources}
field_urls = {url for fields in answer.get('field_assessments', {}).values() for field in fields.values() for url in field.get('sources', [])}
reasoning_urls = {url for url in saved_urls if url and url in str(answer.get('reasoning') or '') + str(reasoning or '')}
citations = sorted(field_urls | reasoning_urls)
display(Markdown('## Cited sources'))
display(HTML(''.join(f'<p><a href="{escape(url, quote=True)}" target="_blank">{escape(url)}</a></p>' for url in citations) or '<p>No saved source URLs cited in the valid answer or API reasoning.</p>'))
recommendation = json.loads((repo_root / demo_dir / 'recommendation.json').read_text(encoding='utf-8'))
display(Markdown('## Recommendation'))
chosen = recommendation.get('recommendation')
if chosen and chosen['speculative_decoding']['usable']:
    pair = chosen['speculative_decoding']
    display(Markdown(f'**{chosen["model_id"]}** with **{pair["draft_model"]}** via **{pair["runtime"]}**. Estimated memory: **{chosen["estimated_bf16_memory_gb"]:.1f} GB** of 80 GB. A100 speedup: **{pair["a100_performance"]}**. [Pair evidence]({pair["source_url"]}).'))
else:
    display(Markdown(f'No eligible recommendation. Blockers: `{recommendation.get("evaluated_candidates", [{}])[0].get("blocking_reasons", [])}`'))
print('Finish reason:', response.get('finish_reason'))

## Nemotron reasoning

We need produce final answer JSON with fields per model. Determine viability: Need speculative decoding head present? Evidence shows draft pair exists (z-lab/Qwen3.5-4B-DFlash). So target has speculative decoding head via external draft. Task fit: needs one 80GB A100 GPU, budget $5/hr, max 2000 tokens. Model size 4B params ~8.68GB download, fits. License Apache-2.0 ok. So viable.

Now fill assessments:

base_model: from HF evidence base_model: null? Actually card excerpt shows base_model: - Qwen/Qwen3.5-4B-Base. So base_model is that. Status verified.

parameter_count: HF shows parameter_count_billion 4.66 -> value 4.66B. Verified.

config: HF config dict given. Verified.

chat_template: present true, length 7756, sha256 given. Return presence, length, sha256 summary. Value maybe object? We'll give string summarizing. Status verified.

license: HF license apache-2.0. Verified.

download_size: HF download_size_gb 8.68. Verified.

All verified.

Now construct JSON: viable_model_ids list with model ID. insufficient_evidence false. reasoning brief. field_assessments dict.

Let's craft.

## Final answer

```json
{
  "viable_model_ids": [
    "Qwen/Qwen3.5-4B"
  ],
  "insufficient_evidence": false,
  "reasoning": "Model fits 80GB A100, download size 8.68GB, license Apache-2.0, speculative decoding head available via z-lab/Qwen3.5-4B-DFlash draft paired with target, enabling speculative decoding within budget.",
  "field_assessments": {
    "Qwen/Qwen3.5-4B": {
      "base_model": {
        "value": "Qwen/Qwen3.5-4B-Base",
        "status": "verified",
        "confidence": 0.95,
        "sources": [
          "https://huggingface.co/Qwen/Qwen3.5-4B/tree/851bf6e806efd8d0a36b00ddf55e13ccb7b8cd0a"
        ],
        "action": "accept"
      },
      "parameter_count": {
        "value": 4.66,
        "status": "verified",
        "confidence": 0.95,
        "sources": [
          "https://huggingface.co/Qwen/Qwen3.5-4B/tree/851bf6e806efd8d0a36b00ddf55e13ccb7b8cd0a"
        ],
        "action": "accept"
      },
      "config": {
        "value": {
          "model_type": "qwen3_5",
          "architectures": [
            "Qwen3_5ForConditionalGeneration"
          ],
          "max_position_embeddings": null
        },
        "status": "verified",
        "confidence": 0.95,
        "sources": [
          "https://huggingface.co/Qwen/Qwen3.5-4B/tree/851bf6e806efd8d0a36b00ddf55e13ccb7b8cd0a"
        ],
        "action": "accept"
      },
      "chat_template": {
        "value": {
          "present": true,
          "length": 7756,
          "sha256": "a4aee8afcf2e0711942cf848899be66016f8d14a889ff9ede07bca099c28f715"
        },
        "status": "verified",
        "confidence": 0.95,
        "sources": [
          "https://huggingface.co/Qwen/Qwen3.5-4B/tree/851bf6e806efd8d0a36b00ddf55e13ccb7b8cd0a"
        ],
        "action": "accept"
      },
      "license": {
        "value": "apache-2.0",
        "status": "verified",
        "confidence": 0.95,
        "sources": [
          "https://huggingface.co/Qwen/Qwen3.5-4B/tree/851bf6e806efd8d0a36b00ddf55e13ccb7b8cd0a"
        ],
        "action": "accept"
      },
      "download_size": {
        "value": 8.68,
        "status": "verified",
        "confidence": 0.95,
        "sources": [
          "https://huggingface.co/Qwen/Qwen3.5-4B/tree/851bf6e806efd8d0a36b00ddf55e13ccb7b8cd0a"
        ],
        "action": "accept"
      }
    }
  }
}
```

## Cited sources

## Recommendation

**Qwen/Qwen3.5-4B** with **z-lab/Qwen3.5-4B-DFlash** via **sglang**. Estimated memory: **42.0 GB** of 80 GB. A100 speedup: **unmeasured**. [Pair evidence](https://huggingface.co/z-lab/Qwen3.5-4B-DFlash).

Finish reason: stop


## Discover models from a task request

Set `model_id = None` for this flow. The CLI extracts constraints from the original request, then searches recent Hugging Face models, the reference model family, and Gemma. It reuses the Hub eligibility filter and score before Tavily searches and extracts source pages. Set the GPU hourly cost explicitly: a budget does not establish the rental price.

In [2]:
sys.path.insert(0, str(repo_root / "src"))
from hf_model_watch.recommender.discovery import discovery_inputs, extract_constraints

model_id = None
discovery_request = (
    "Find a new model like Qwen3.5-4B with optional speculative decoding head "
    "for one 80GB A100 GPU, a budget of $5/hour, and a maximum of 2000 generated tokens; "
    "any license, sweep mode."
)
deployment_inputs = {"gpu_hourly_cost_usd": 5}
values, missing = discovery_inputs(discovery_request, deployment_inputs)
print(json.dumps(extract_constraints(discovery_request), indent=2))
print("Missing inputs:", missing)
assert not missing, "Supply missing deployment inputs before running discovery"

{
  "reference_model": "Qwen3.5-4B",
  "family": "Qwen",
  "target_params_billion": 4.0,
  "features": [
    "speculative decoding"
  ],
  "use_case": "one 80GB A100 GPU",
  "gpu_type": "A100",
  "gpu_memory_gb": 80.0,
  "budget_usd_per_hour": 5.0,
  "gpu_hourly_cost_usd": null,
  "max_generation_tokens": 2000,
  "license": "any",
  "framework_mode": "sweep",
  "framework": null,
  "recent": true
}
Missing inputs: []


/home/maorb/.venvs/nemotron_runner/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
import subprocess
import shlex
import tempfile
import time

# Use the same Python runner as the explicit-model cells above.
discovery_dir = Path('artifacts/recommendations') / f"discovery-{datetime.now(timezone.utc):%Y%m%d-%H%M%S}"
command = shlex.split(python_runner) + [
    '-c', 'from hf_model_watch.cli import app; app()', 'recommend', discovery_request,
    '--gpu-memory-gb', str(values['gpu_memory_gb']),
    '--budget-usd-per-hour', str(values['budget_usd_per_hour']),
    '--gpu-hourly-cost-usd', str(values['gpu_hourly_cost_usd']),
    '--license', values['license'], '--framework-mode', values['framework_mode'],
    '--output-dir', discovery_dir.as_posix(), '--json',
]
# Keep JSON on stdout while displaying CLI progress from stderr as it arrives.
started = time.monotonic()
with tempfile.TemporaryFile(mode='w+t', encoding='utf-8') as output_file:
    process = subprocess.Popen(
        command, cwd=repo_root, stdout=output_file, stderr=subprocess.PIPE,
        text=True, bufsize=1,
    )
    for line in process.stderr:
        print(f"[{time.monotonic() - started:5.1f}s] {line.rstrip()}", flush=True)
    returncode = process.wait()
    output_file.seek(0)
    response_text = output_file.read()
if returncode:
    raise RuntimeError(response_text or f'Discovery exited with code {returncode}; check progress above')
artifact = repo_root / discovery_dir / 'discovery.json'
if not artifact.exists():
    raise RuntimeError(
        f'Discovery finished without a saved artifact. '
        f'CLI output: {response_text[:500]!r}; runner: {command[:4]!r}'
    )
discovery_result = json.loads(artifact.read_text(encoding='utf-8'))
discovery_result['artifact_path'] = str(artifact)
print('Saved:', discovery_result['artifact_path'])
print('Status:', discovery_result['status'])

[  0.8s] Scanning recent Hugging Face models...
[  0.8s] TRACE_EVENT {"event_type":"scan_started","message":"Hugging Face model scan started","metadata":{"output_path":"artifacts/recommendations/discovery-20260918-115446/recent.jsonl"},"stage":"scan","status":"started","timestamp":"2026-09-18T11:54:46.899073+00:00"}
[ 14.8s] TRACE_EVENT {"artifacts":{"output_path":"artifacts/recommendations/discovery-20260918-115446/recent.jsonl"},"event_type":"scan_completed","message":"Hugging Face model scan completed","metrics":{"models_written":36},"stage":"scan","status":"completed","timestamp":"2026-09-18T11:55:00.912316+00:00"}
[ 14.8s] Recent scan returned 36 models.
[ 14.8s] Searching Qwen family...
[ 21.7s] Searching Gemma family...
[ 30.9s] Scoring and filtering Hub candidates...
[ 30.9s] recent_similar: 36 models found.
[ 30.9s] same_family: 25 models found.
[ 30.9s] gemma: 25 models found.
[ 30.9s] Selected 3 candidates for task-fit research.
[ 30.9s] Tavily Search: ReliquaryForge/qwen3-4

In [4]:
from IPython.display import Markdown, display

artifact_path = Path(discovery_result.get('artifact_path') or repo_root / discovery_dir / 'discovery.json')
if artifact_path.exists():
    saved_at = datetime.fromtimestamp(artifact_path.stat().st_mtime, timezone.utc).astimezone()
    print(f'Results from: {artifact_path}')
    print(f'Saved at: {saved_at:%Y-%m-%d %H:%M:%S %Z}')
    source_path = repo_root / 'src/hf_model_watch/recommender/discovery.py'
    if source_path.stat().st_mtime > artifact_path.stat().st_mtime:
        print('Discovery code is newer than this artifact; rerun the discovery cell above.')
recommendations = discovery_result.get('recommendations', [])
for strategy, summary in discovery_result.get('scan_diagnostics', {}).items():
    blockers = sorted(summary.get('blockers', {}).items(), key=lambda item: -item[1])[:3]
    print(f"{strategy}: {summary['found']} found, {summary['size_matched']} near target size, {summary['eligible']} Hub-eligible")
    if blockers:
        print('  Main blockers:', ', '.join(f'{reason} ({count})' for reason, count in blockers))
for number, item in enumerate(recommendations, 1):
    title = ' **(strongest recommendation)**' if number == 1 else ''
    lines = [f"### {number}. {item['model_id']}{title}",
             '', '**Decision reasoning**',
             item.get('reasoning') or 'This saved result predates decision reasoning. Rerun the discovery cell above.',
             '', f"HF score: {item['hf_score']:.3f}",
             '', '**Why it fits**']
    lines += [f"- {reason}" for reason in item['why_it_fits']]
    lines += ['', '**Task fit**']
    lines += [f"- {name.replace('_', ' ')}: {status}" for name, status in item['task_fit'].items()]
    lines += ['', '**Potential limitations**']
    lines += [f"- {reason}" for reason in item['limitations']] or ['- None recorded']
    lines += ['', '**Evidence URLs**']
    lines += [f"- {source['source_url']}" for source in item['evidence'] if source.get('source_url')]
    display(Markdown('\n'.join(lines)))
if not recommendations:
    display(Markdown('No viable recommendations were found. Inspect `discovery.json` for the extracted constraints and status.'))

Results from: /mnt/c/Users/maorb/work/hf_validator/artifacts/recommendations/discovery-20260918-115446/discovery.json
Saved at: 2026-09-18 14:55:19 IDT
recent_similar: 36 found, 13 near target size, 4 Hub-eligible
  Main blockers: missing_config (8), missing_license (8), missing_download_size (8)
same_family: 25 found, 5 near target size, 3 Hub-eligible
  Main blockers: not_causal_lm (2)
gemma: 25 found, 10 near target size, 0 Hub-eligible
  Main blockers: not_causal_lm (10), unsupported_finetune_format (6), missing_tokenizer (3)


### 1. Qwen/Qwen3-4B **(strongest recommendation)**

**Decision reasoning**
The existing Hugging Face scanner marked Qwen/Qwen3-4B eligible with a score of 0.791. Its estimated 4.022B parameters are near the requested 4B scale. The estimated serving footprint is 38.476 GB against 80 GB of GPU memory; this is a capacity estimate. The supplied GPU cost of $5/hour is within the $5/hour budget. Speculative decoding is documented for this exact target with external draft z-lab/Qwen3-4B-DFlash-b16 via sglang (https://huggingface.co/z-lab/Qwen3-4B-DFlash-b16?local-app=sglang); speed on the requested GPU is unmeasured. Remaining uncertainty: Prompt length is unspecified; context fit is an estimate; Documented external draft pair; A100 speedup is unmeasured.

HF score: 0.791

**Why it fits**
- Documented external speculative draft: z-lab/Qwen3-4B-DFlash-b16 via sglang
- HF scanner score 0.791
- Approximately 4.022B parameters
- Fits the stated 80 GB memory estimate

**Task fit**
- speculative decoding: verified
- gpu fit: inferred
- framework fit: inferred

**Potential limitations**
- Prompt length is unspecified; context fit is an estimate
- Documented external draft pair; A100 speedup is unmeasured

**Evidence URLs**
- https://www.jetson-ai-lab.com/tutorials/speculative-decoding
- https://llm.co/llms/qwen3-4b-dflash-b16
- https://llm.co/llms/qwen3-4b-dflash-b16
- https://github.com/vllm-project/vllm/issues/45669
- https://huggingface.co/z-lab/Qwen3-4B-DFlash-b16?local-app=sglang
- https://huggingface.co/z-lab/Qwen3-4B-DFlash-b16?library=transformers
- https://huggingface.co/z-lab/Qwen3-4B-DFlash-b16?local-app=docker-model-runner
- https://huggingface.co/z-lab/Qwen3-4B-DFlash-b16?ref=pytorchkr
- https://huggingface.co/z-lab/Qwen3-4B-DFlash-b16
- https://huggingface.co/z-lab/Qwen3-4B-DFlash-b16

### 2. ReliquaryForge/qwen3-4b-base-dapo-v4

**Decision reasoning**
The existing Hugging Face scanner marked ReliquaryForge/qwen3-4b-base-dapo-v4 eligible with a score of 0.822. Its estimated 4.022B parameters are near the requested 4B scale. The estimated serving footprint is 23.492 GB against 80 GB of GPU memory; this is a capacity estimate. The supplied GPU cost of $5/hour is within the $5/hour budget. Evidence for speculative decoding is unknown. Remaining uncertainty: Prompt length is unspecified; context fit is an estimate; No Tavily evidence found for the task request; Speculative decoding support is unverified.

HF score: 0.822

**Why it fits**
- HF scanner score 0.822
- Approximately 4.022B parameters
- Fits the stated 80 GB memory estimate

**Task fit**
- speculative decoding: unknown
- gpu fit: inferred
- framework fit: inferred

**Potential limitations**
- Prompt length is unspecified; context fit is an estimate
- No Tavily evidence found for the task request
- Speculative decoding support is unverified

**Evidence URLs**

In [5]:
# Change this number to select a different recommendation, then rerun this cell.
selected_number = 1

print('Selection cell started. No network calls or benchmarking run here.', flush=True)
if not recommendations:
    print('No recommendations to select. Check the scan diagnostics above.', flush=True)
else:
    print('Available models:', flush=True)
    for number, item in enumerate(recommendations, 1):
        print(f"  {number}. {item['model_id']} (HF score {item['hf_score']:.3f})", flush=True)
    if selected_number not in range(1, len(recommendations) + 1):
        raise ValueError(f'Select a number from 1 to {len(recommendations)}')
    selected_model_id = recommendations[selected_number - 1]['model_id']
    print(f'Selected: {selected_model_id}', flush=True)
    discovery_result['selected_model_id'] = selected_model_id
    artifact = repo_root / discovery_dir / 'discovery.json'
    artifact.write_text(json.dumps(discovery_result, indent=2) + '\n', encoding='utf-8')
    print(f'Saved selection to: {artifact}', flush=True)
    display(Markdown(
        f"**Selected model:** `{selected_model_id}`\n\n"
        "[ Optimize Inference + Benchmark ] (placeholder)"
    ))

Selection cell started. No network calls or benchmarking run here.
Available models:
  1. Qwen/Qwen3-4B (HF score 0.791)
  2. ReliquaryForge/qwen3-4b-base-dapo-v4 (HF score 0.822)
Selected: Qwen/Qwen3-4B
Saved selection to: /mnt/c/Users/maorb/work/hf_validator/artifacts/recommendations/discovery-20260918-115446/discovery.json


**Selected model:** `Qwen/Qwen3-4B`

[ Optimize Inference + Benchmark ] (placeholder)

## Run the interactive terminal UI

From the repository root, run:

```bash
PYTHONPATH=src python -m hf_model_watch recommend "Find a new model like Qwen3.5-4B with optional speculative decoding head"
```

The command prompts only for deployment values missing from the request, lists up to three models, and asks you to select one. Add `--gpu-memory-gb`, `--budget-usd-per-hour`, `--gpu-hourly-cost-usd`, `--license`, and `--framework-mode` to skip those prompts. Run this command in WSL with the `nemotron_runner` environment active. It uses the repository source directly and does not install GPU packages. The selection is saved in `discovery.json`; the optimization button is a placeholder.